In [ ]:
# ============================================================
# DAILY SIGNAL DASHBOARD — Pip Installs
# ============================================================
# Uncomment and run if needed:
#
# !pip install yfinance pandas numpy ta-lib
# !pip install matplotlib


In [ ]:
import sys, os
# Ensure lib/ is importable (for Google Sheets logging)
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

# ============================================================
# IMPORTS + INLINE HELPER FUNCTIONS
# ============================================================
import numpy as np
import pandas as pd
import warnings
import datetime
warnings.filterwarnings('ignore')

try:
    import talib
except ImportError:
    print("TA-Lib not available. Using pandas-based fallbacks.")
    talib = None

# ────────────────────────────────────────────────────────────
# Regime Filter (inlined from lib/regime_filter.py)
# ────────────────────────────────────────────────────────────

def realized_volatility(close, window=21):
    """Annualized realized volatility over trailing window. Uses log returns."""
    log_rets = np.log(close / close.shift(1))
    return log_rets.rolling(window=window).std() * np.sqrt(252)


def classify_regimes(close, trend_period=200, vol_window=21, vol_lookback=126):
    """
    Classify each day into a regime using ONLY backward-looking data.
    All indicators shifted by 1 so regime on day T uses day T-1 close.
    """
    close = close.astype(float).squeeze()
    close.name = 'close'

    sma = close.rolling(window=trend_period).mean()
    prev_close = close.shift(1)
    prev_sma = sma.shift(1)
    trend = pd.Series(np.where(prev_close > prev_sma, 'BULL', 'BEAR'),
                       index=close.index, name='trend')

    rvol = realized_volatility(close, window=vol_window).shift(1)
    vol_median = rvol.rolling(window=vol_lookback).median()
    vol_regime = pd.Series(np.where(rvol > vol_median, 'HIGH', 'LOW'),
                            index=close.index, name='vol_regime')

    regime = pd.Series(
        np.where(
            (trend == 'BULL') & (vol_regime == 'LOW'), 'BULL_CALM',
            np.where(
                (trend == 'BULL') & (vol_regime == 'HIGH'), 'BULL_WILD',
                np.where(
                    (trend == 'BEAR') & (vol_regime == 'LOW'), 'BEAR_CALM',
                    'BEAR_WILD'
                )
            )
        ),
        index=close.index,
        name='regime'
    )

    regime_code = regime.map({
        'BULL_CALM': 0, 'BULL_WILD': 1,
        'BEAR_CALM': 2, 'BEAR_WILD': 3
    }).astype(float)

    return pd.DataFrame({
        'close': close,
        'sma': sma,
        'trend': trend,
        'realized_vol': rvol,
        'vol_median': vol_median,
        'vol_regime': vol_regime,
        'regime': regime,
        'regime_code': regime_code
    })


# ────────────────────────────────────────────────────────────
# TA-Lib fallback helpers (used when talib is not installed)
# ────────────────────────────────────────────────────────────

def _sma(series, period):
    return series.rolling(window=period).mean()

def _ema(series, period):
    return series.ewm(span=period, adjust=False).mean()

def _rsi(close, period=14):
    delta = close.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def _atr(high, low, close, period=14):
    tr1 = high - low
    tr2 = (high - close.shift(1)).abs()
    tr3 = (low - close.shift(1)).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    return tr.rolling(window=period).mean()

def _max(series, period):
    return series.rolling(window=period).max()

def _min(series, period):
    return series.rolling(window=period).min()


# ────────────────────────────────────────────────────────────
# Signal Generation Functions
# All return (entries, exits) as boolean pd.Series
# All apply 1-bar execution delay (shift raw signals by 1)
# ────────────────────────────────────────────────────────────

def macd_signals(close, fast=12, slow=26, signal_p=9):
    """MACD crossover signals with 1-bar execution delay."""
    if talib is not None:
        macd_line, signal_line, hist = talib.MACD(close.values, fastperiod=fast, slowperiod=slow, signalperiod=signal_p)
        macd_s = pd.Series(macd_line, index=close.index)
        signal_s = pd.Series(signal_line, index=close.index)
    else:
        fast_ema = _ema(close, fast)
        slow_ema = _ema(close, slow)
        macd_s = fast_ema - slow_ema
        signal_s = _ema(macd_s, signal_p)

    entries_raw = (macd_s.shift(1) <= signal_s.shift(1)) & (macd_s > signal_s)
    exits_raw = (macd_s.shift(1) >= signal_s.shift(1)) & (macd_s < signal_s)

    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)

    indicator_values = macd_s - signal_s  # MACD histogram
    return entries, exits, {'macd_hist': indicator_values, 'macd': macd_s, 'signal': signal_s}


def rsi_signals(close, rsi_len=14, oversold=30, overbought=70):
    """RSI mean-reversion signals with 1-bar execution delay."""
    if talib is not None:
        rsi_vals = talib.RSI(close.values, timeperiod=rsi_len)
        rsi_s = pd.Series(rsi_vals, index=close.index)
    else:
        rsi_s = _rsi(close, rsi_len)

    entries_raw = (rsi_s.shift(1) <= oversold) & (rsi_s > oversold)
    exits_raw = (rsi_s.shift(1) <= overbought) & (rsi_s > overbought)

    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)

    return entries, exits, {'rsi': rsi_s}


def ema_signals(close, fast=9, slow=21, trend=50):
    """EMA crossover + trend filter signals with 1-bar execution delay."""
    if talib is not None:
        fast_ema = pd.Series(talib.EMA(close.values, timeperiod=fast), index=close.index)
        slow_ema = pd.Series(talib.EMA(close.values, timeperiod=slow), index=close.index)
        trend_sma = pd.Series(talib.SMA(close.values, timeperiod=trend), index=close.index)
    else:
        fast_ema = _ema(close, fast)
        slow_ema = _ema(close, slow)
        trend_sma = _sma(close, trend)

    entries_raw = ((fast_ema.shift(1) <= slow_ema.shift(1)) & (fast_ema > slow_ema) & (close > trend_sma))
    exits_raw = ((fast_ema.shift(1) >= slow_ema.shift(1)) & (fast_ema < slow_ema))

    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)

    return entries, exits, {'fast_ema': fast_ema, 'slow_ema': slow_ema, 'trend_sma': trend_sma}


def donchian_signals(high, low, close, entry_p=20, exit_p=10, filter_p=50):
    """Donchian channel breakout signals with 1-bar execution delay."""
    if talib is not None:
        upper_channel = pd.Series(talib.MAX(high.values, timeperiod=entry_p), index=close.index).shift(1)
        lower_channel = pd.Series(talib.MIN(low.values, timeperiod=exit_p), index=close.index).shift(1)
        trend_filter = pd.Series(talib.SMA(close.values, timeperiod=filter_p), index=close.index).shift(1)
    else:
        upper_channel = _max(high, entry_p).shift(1)
        lower_channel = _min(low, exit_p).shift(1)
        trend_filter = _sma(close, filter_p).shift(1)

    entries_raw = (close > upper_channel) & (close > trend_filter)
    exits_raw = (close < lower_channel)

    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)

    return entries, exits, {'upper': upper_channel, 'lower': lower_channel, 'trend_filter': trend_filter}


def bollinger_signals(close, period=20, std_dev=2.0, exit_type='middle'):
    """Bollinger Band mean-reversion signals with 1-bar execution delay."""
    if talib is not None:
        bb_upper, bb_middle, bb_lower = talib.BBANDS(close.values, timeperiod=period, nbdevup=std_dev, nbdevdn=std_dev)
        bb_upper_s = pd.Series(bb_upper, index=close.index)
        bb_middle_s = pd.Series(bb_middle, index=close.index)
        bb_lower_s = pd.Series(bb_lower, index=close.index)
    else:
        bb_middle_s = _sma(close, period)
        rolling_std = close.rolling(window=period).std()
        bb_upper_s = bb_middle_s + (std_dev * rolling_std)
        bb_lower_s = bb_middle_s - (std_dev * rolling_std)

    # Entry: price crosses below lower band (mean-reversion buy)
    entries_raw = (close.shift(1) >= bb_lower_s.shift(1)) & (close < bb_lower_s)

    # Exit: price crosses above middle or upper band
    if exit_type == 'upper':
        exits_raw = (close.shift(1) <= bb_upper_s.shift(1)) & (close > bb_upper_s)
    else:
        exits_raw = (close.shift(1) <= bb_middle_s.shift(1)) & (close > bb_middle_s)

    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)

    return entries, exits, {'bb_upper': bb_upper_s, 'bb_middle': bb_middle_s, 'bb_lower': bb_lower_s}


def atr_signals(high, low, close, atr_period=14, sma_period=20, atr_mult=2.0):
    """ATR volatility breakout signals with 1-bar execution delay."""
    if talib is not None:
        atr_vals = pd.Series(talib.ATR(high.values, low.values, close.values, timeperiod=atr_period), index=close.index)
        sma_vals = pd.Series(talib.SMA(close.values, timeperiod=sma_period), index=close.index)
    else:
        atr_vals = _atr(high, low, close, atr_period)
        sma_vals = _sma(close, sma_period)

    upper_band = sma_vals + (atr_mult * atr_vals)
    lower_band = sma_vals - (atr_mult * atr_vals)

    entries_raw = (close > upper_band)
    exits_raw = (close < lower_band)

    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)

    return entries, exits, {'atr': atr_vals, 'upper_band': upper_band, 'lower_band': lower_band, 'sma': sma_vals}


# ────────────────────────────────────────────────────────────
# Strategy dispatcher
# ────────────────────────────────────────────────────────────

STRATEGY_MAP = {
    'MACD':      lambda d: macd_signals(d['close'], **{k: v for k, v in d['params'].items() if k in ('fast_period', 'slow_period', 'signal_period')}) if 'fast_period' in d['params'] else macd_signals(d['close'], **d['params']),
    'RSI':       lambda d: rsi_signals(d['close'], **d['params']),
    'EMA':       lambda d: ema_signals(d['close'], **d['params']),
    'Donchian':  lambda d: donchian_signals(d['high'], d['low'], d['close'], **d['params']),
    'Bollinger': lambda d: bollinger_signals(d['close'], **d['params']),
    'ATR':       lambda d: atr_signals(d['high'], d['low'], d['close'], **d['params']),
}


def dispatch_strategy(strategy_name, data_dict):
    """Call the right signal function based on strategy name."""
    strat = strategy_name.upper().replace(' ', '')
    # Normalize names
    name_map = {
        'MACD': 'MACD', 'RSI': 'RSI', 'EMA': 'EMA',
        'DONCHIAN': 'Donchian', 'BOLLINGER': 'Bollinger', 'ATR': 'ATR'
    }
    key = name_map.get(strat, strategy_name)

    # Build the argument dict
    arg = {
        'close': data_dict['close'],
        'high': data_dict.get('high'),
        'low': data_dict.get('low'),
        'params': data_dict['info']['params']
    }

    # Handle MACD param name mapping (fast_period -> fast, etc.)
    if key == 'MACD':
        p = arg['params']
        return macd_signals(arg['close'],
                           fast=p.get('fast_period', p.get('fast', 12)),
                           slow=p.get('slow_period', p.get('slow', 26)),
                           signal_p=p.get('signal_period', p.get('signal_p', 9)))
    elif key == 'EMA':
        p = arg['params']
        return ema_signals(arg['close'],
                          fast=p.get('fast', 9),
                          slow=p.get('slow', 21),
                          trend=p.get('trend', 50))
    elif key == 'RSI':
        return rsi_signals(arg['close'], **arg['params'])
    elif key == 'Donchian':
        p = arg['params']
        return donchian_signals(arg['high'], arg['low'], arg['close'],
                               entry_p=p.get('entry_period', 20),
                               exit_p=p.get('exit_period', 10),
                               filter_p=p.get('filter_period', 50))
    elif key == 'Bollinger':
        return bollinger_signals(arg['close'], **arg['params'])
    elif key == 'ATR':
        return atr_signals(arg['high'], arg['low'], arg['close'], **arg['params'])
    else:
        raise ValueError(f"Unknown strategy: {strategy_name}")


print("All imports and helper functions loaded.")
print(f"Available strategies: {list(STRATEGY_MAP.keys())}")


In [ ]:
# ============================================================
# DAILY SIGNAL DASHBOARD — CONFIGURATION
# ============================================================
#
# Option 1: Set your portfolio manually
# Option 2: Paste the output from Quantitative_Momentum_FTMO.ipynb

PORTFOLIO = {
    # ticker: {name, strategy, params}
    # These are DEFAULT recommendations — replace with your QM notebook output
    '^IXIC':   {'name': 'US100',    'strategy': 'MACD',     'params': {'fast_period': 12, 'slow_period': 26, 'signal_period': 9}},
    'GC=F':    {'name': 'XAUUSD',   'strategy': 'Donchian', 'params': {'entry_period': 20, 'exit_period': 10, 'filter_period': 50}},
    'BTC-USD': {'name': 'BTCUSD',   'strategy': 'Donchian', 'params': {'entry_period': 25, 'exit_period': 12, 'filter_period': 75}},
    'EURUSD=X':{'name': 'EURUSD',   'strategy': 'RSI',      'params': {'rsi_len': 14, 'oversold': 30, 'overbought': 70}},
    'ETH-USD': {'name': 'ETHUSD',   'strategy': 'Donchian', 'params': {'entry_period': 25, 'exit_period': 12, 'filter_period': 75}},
    '^GDAXI':  {'name': 'DAX',      'strategy': 'MACD',     'params': {'fast_period': 12, 'slow_period': 26, 'signal_period': 9}},
    'CL=F':    {'name': 'Crude Oil', 'strategy': 'Donchian', 'params': {'entry_period': 20, 'exit_period': 10, 'filter_period': 50}},
    'GBPUSD=X':{'name': 'GBPUSD',   'strategy': 'RSI',      'params': {'rsi_len': 14, 'oversold': 30, 'overbought': 70}},
}

ACCOUNT_SIZE = 100_000
RISK_PER_TRADE = 0.01   # 1% risk per trade
LOOKBACK_DAYS = 400     # Days of data to download

import datetime
print(f"DAILY SIGNAL DASHBOARD")
print(f"Date: {datetime.date.today()}")
print(f"Portfolio: {len(PORTFOLIO)} instruments")
print(f"Account: ${ACCOUNT_SIZE:,}")


In [ ]:
# ============================================================
# DOWNLOAD LATEST DATA
# ============================================================
import yfinance as yf

all_data = {}
print("Downloading latest data...\n")

for ticker, info in PORTFOLIO.items():
    try:
        df = yf.download(ticker, period=f'{LOOKBACK_DAYS}d', interval='1d', progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            close = df[('Close', ticker)].astype(float).squeeze()
            high = df[('High', ticker)].astype(float).squeeze()
            low = df[('Low', ticker)].astype(float).squeeze()
        else:
            close = df['Close'].astype(float).squeeze()
            high = df['High'].astype(float).squeeze()
            low = df['Low'].astype(float).squeeze()
        
        all_data[ticker] = {'close': close, 'high': high, 'low': low, 'info': info}
        print(f"  {info['name']:12s} | {len(close)} bars | Last: {close.iloc[-1]:.2f} | {close.index[-1].date()}")
    except Exception as e:
        print(f"  {info['name']:12s} | FAILED: {e}")

print(f"\nLoaded {len(all_data)}/{len(PORTFOLIO)} instruments")


In [ ]:
# ============================================================
# GENERATE SIGNALS — THE KEY CELL
# ============================================================
# For each instrument:
#   1. Compute strategy signals using assigned params
#   2. Check current regime
#   3. Determine if there's an active signal today
#   4. Check if currently in a position (last entry without exit)

# Regime filter rules:
#   - Trend-following (MACD, EMA, Donchian, ATR): BLOCKED in BEAR_WILD
#   - Mean-reversion (RSI, Bollinger): BLOCKED in BULL_WILD or BEAR_WILD

TREND_FOLLOWING = {'MACD', 'EMA', 'Donchian', 'ATR'}
MEAN_REVERSION  = {'RSI', 'Bollinger'}

dashboard = {}

for ticker, data in all_data.items():
    info = data['info']
    strategy = info['strategy']
    name = info['name']

    try:
        # 1. Generate signals
        entries, exits, indicators = dispatch_strategy(strategy, data)

        # 2. Classify regime
        regime_df = classify_regimes(data['close'])
        current_regime = regime_df['regime'].dropna().iloc[-1]

        # 3. Regime filter
        if strategy in TREND_FOLLOWING:
            regime_blocked = current_regime == 'BEAR_WILD'
        elif strategy in MEAN_REVERSION:
            regime_blocked = current_regime in ('BULL_WILD', 'BEAR_WILD')
        else:
            regime_blocked = False

        regime_filter = 'BLOCKED' if regime_blocked else 'PASS'

        # 4. Determine position state by scanning entries/exits chronologically
        in_position = False
        last_entry_date = None
        last_entry_price = None

        valid_entries = entries.dropna()
        valid_exits = exits.dropna()

        for dt in entries.index:
            if entries.get(dt, False):
                in_position = True
                last_entry_date = dt
                last_entry_price = data['close'].get(dt, None)
            if exits.get(dt, False) and in_position:
                in_position = False

        # 5. Today's signal
        today_entry = entries.iloc[-1] if len(entries) > 0 else False
        today_exit = exits.iloc[-1] if len(exits) > 0 else False

        if today_entry and not regime_blocked:
            current_signal = 'BUY'
        elif today_entry and regime_blocked:
            current_signal = 'SKIP'
        elif today_exit:
            current_signal = 'EXIT'
        else:
            current_signal = 'HOLD'

        # 6. Signal strength
        if current_signal in ('BUY', 'EXIT'):
            if current_regime in ('BULL_CALM',) and strategy in TREND_FOLLOWING:
                signal_strength = 'STRONG'
            elif current_regime in ('BEAR_CALM',) and strategy in MEAN_REVERSION:
                signal_strength = 'STRONG'
            elif current_regime in ('BULL_WILD',) and strategy in TREND_FOLLOWING:
                signal_strength = 'WEAK'
            else:
                signal_strength = 'WEAK'
        else:
            signal_strength = '-'

        # 7. Current price and P&L if in position
        current_price = data['close'].iloc[-1]
        pnl_pct = None
        if in_position and last_entry_price is not None:
            pnl_pct = (current_price - last_entry_price) / last_entry_price

        # Collect indicator snapshot
        indicator_snapshot = {}
        for k, v in indicators.items():
            if isinstance(v, pd.Series) and len(v) > 0:
                last_val = v.dropna()
                indicator_snapshot[k] = last_val.iloc[-1] if len(last_val) > 0 else None

        dashboard[ticker] = {
            'name': name,
            'strategy': strategy,
            'current_signal': current_signal,
            'in_position': in_position,
            'regime': current_regime,
            'regime_filter': regime_filter,
            'signal_strength': signal_strength,
            'last_entry_date': last_entry_date,
            'last_entry_price': last_entry_price,
            'price': current_price,
            'pnl_pct': pnl_pct,
            'indicators': indicator_snapshot,
            'entries': entries,
            'exits': exits,
            'indicator_series': indicators,
            'regime_df': regime_df,
        }

    except Exception as e:
        print(f"  ERROR processing {name} ({ticker}): {e}")
        import traceback
        traceback.print_exc()

print(f"Signals generated for {len(dashboard)}/{len(all_data)} instruments")


In [ ]:
# ============================================================
# DAILY TRADING BRIEFING — MAIN OUTPUT
# ============================================================
import datetime

today_str = datetime.date.today().strftime('%B %d, %Y')

print("=" * 70)
print(f"  DAILY TRADING BRIEFING — {today_str}")
print("=" * 70)

# ── MARKET REGIME OVERVIEW ──────────────────────────────────
print("\n  MARKET REGIME OVERVIEW:")

regime_descriptions = {
    'BULL_CALM': 'Trending up, low vol. Full size.',
    'BULL_WILD': 'Trending up, high vol. Reduce size.',
    'BEAR_CALM': 'Trending down, low vol. Mean-revert or flat.',
    'BEAR_WILD': 'Crisis mode, high vol. Stay out.',
}

for ticker, d in dashboard.items():
    regime = d['regime']
    desc = regime_descriptions.get(regime, '')
    print(f"    {d['name']:12s}  {regime:12s} — {desc}")

# ── ACTION ITEMS ────────────────────────────────────────────
print("\n  ACTION ITEMS:")

buy_count = 0
exit_count = 0
blocked_count = 0
active_positions = 0

for ticker, d in dashboard.items():
    sig = d['current_signal']
    name = d['name']
    strategy = d['strategy']
    regime = d['regime']
    strength = d['signal_strength']
    in_pos = d['in_position']

    if in_pos:
        active_positions += 1

    if sig == 'BUY':
        buy_count += 1
        # Build indicator detail
        if strategy == 'MACD':
            detail = 'MACD crossover fired yesterday'
        elif strategy == 'Donchian':
            detail = 'Donchian breakout fired yesterday'
        elif strategy == 'RSI':
            rsi_val = d['indicators'].get('rsi', 0)
            detail = f'RSI crossed above oversold (RSI={rsi_val:.1f})' if rsi_val else 'RSI crossover'
        elif strategy == 'EMA':
            detail = 'EMA crossover + trend confirmed'
        elif strategy == 'Bollinger':
            detail = 'Price touched lower Bollinger Band'
        elif strategy == 'ATR':
            detail = 'ATR breakout above upper band'
        else:
            detail = f'{strategy} entry signal'

        conf = 'CONFIRMED' if strength == 'STRONG' else 'CAUTION — reduce size'
        emoji = '\U0001f7e2'  # green circle
        print(f"    {emoji} BUY  {name:12s} | {detail} | Regime: {regime} ({conf})")

    elif sig == 'EXIT':
        exit_count += 1
        if strategy == 'MACD':
            detail = 'MACD bearish crossover'
        elif strategy == 'RSI':
            rsi_val = d['indicators'].get('rsi', 0)
            detail = f'RSI hit overbought (RSI={rsi_val:.1f})' if rsi_val else 'RSI exit'
        elif strategy == 'Donchian':
            detail = 'Price broke below exit channel'
        elif strategy == 'Bollinger':
            detail = 'Price crossed above middle band'
        elif strategy == 'ATR':
            detail = 'Price broke below lower ATR band'
        else:
            detail = f'{strategy} exit signal'

        emoji = '\U0001f534'  # red circle
        print(f"    {emoji} EXIT {name:12s} | {detail} | Regime: {regime} (CONFIRMED)")

    elif sig == 'SKIP':
        blocked_count += 1
        if strategy == 'MACD':
            detail = 'MACD entry'
        elif strategy == 'Donchian':
            detail = 'Donchian entry'
        else:
            detail = f'{strategy} entry'
        emoji = '\u26d4'  # no entry
        print(f"    {emoji} SKIP {name:12s} | {detail} but regime {regime} — BLOCKED")

    elif sig == 'HOLD':
        emoji = '\u26aa'  # white circle
        if in_pos:
            entry_date = d['last_entry_date']
            entry_str = entry_date.strftime('%b %d') if entry_date is not None else '?'
            pnl = d['pnl_pct']
            pnl_str = f"P&L: {pnl:+.1%}" if pnl is not None else "P&L: N/A"
            print(f"    {emoji} HOLD {name:12s} | In position since {entry_str} | {pnl_str}")
        else:
            # What is it waiting for?
            if strategy == 'Donchian':
                wait = 'Waiting for Donchian breakout'
            elif strategy == 'MACD':
                wait = 'Waiting for MACD crossover'
            elif strategy == 'RSI':
                rsi_val = d['indicators'].get('rsi', 0)
                wait = f'Waiting for RSI oversold (current RSI={rsi_val:.1f})' if rsi_val else 'Waiting for RSI signal'
            elif strategy == 'EMA':
                wait = 'Waiting for EMA crossover'
            elif strategy == 'Bollinger':
                wait = 'Waiting for lower band touch'
            elif strategy == 'ATR':
                wait = 'Waiting for ATR breakout'
            else:
                wait = f'Waiting for {strategy} signal'
            print(f"    {emoji} HOLD {name:12s} | No position | {wait}")

# ── POSITION SUMMARY ───────────────────────────────────────
total_instruments = len(dashboard)
print(f"\n  POSITION SUMMARY:")
print(f"    Active positions: {active_positions}/{total_instruments}")
print(f"    Instruments with signals: {buy_count} buys, {exit_count} exits")
print(f"    Regime-blocked signals: {blocked_count}")

# ── RISK CHECK ──────────────────────────────────────────────
max_alloc = ACCOUNT_SIZE / total_instruments if total_instruments > 0 else 0
pct_invested = active_positions / total_instruments * 100 if total_instruments > 0 else 0
pct_cash = 100 - pct_invested

print(f"\n  RISK CHECK:")
print(f"    Max allocation per instrument: ${max_alloc:,.0f} (1/{total_instruments} of account)")
print(f"    If all positions active: {pct_invested:.1f}% invested, {pct_cash:.1f}% cash")

print("\n" + "=" * 70)


In [ ]:
# ============================================================
# RECENT SIGNAL HISTORY — Last 10 Trading Days
# ============================================================
# For each instrument, show day-by-day signals to see if
# signals just fired or are stale.

HISTORY_DAYS = 10

for ticker, d in dashboard.items():
    name = d['name']
    strategy = d['strategy']
    entries = d['entries']
    exits = d['exits']
    close = all_data[ticker]['close']
    regime_df = d['regime_df']
    ind_series = d['indicator_series']

    print(f"\n{'='*70}")
    print(f"  {name} ({ticker}) — {strategy}")
    print(f"{'='*70}")

    # Determine which indicator column to show
    if strategy == 'MACD':
        ind_label = 'MACD_Hist'
        ind_data = ind_series.get('macd_hist', pd.Series(dtype=float))
    elif strategy == 'RSI':
        ind_label = 'RSI'
        ind_data = ind_series.get('rsi', pd.Series(dtype=float))
    elif strategy == 'Donchian':
        ind_label = 'Upper_Ch'
        ind_data = ind_series.get('upper', pd.Series(dtype=float))
    elif strategy == 'EMA':
        ind_label = 'Fast_EMA'
        ind_data = ind_series.get('fast_ema', pd.Series(dtype=float))
    elif strategy == 'Bollinger':
        ind_label = 'BB_Lower'
        ind_data = ind_series.get('bb_lower', pd.Series(dtype=float))
    elif strategy == 'ATR':
        ind_label = 'ATR'
        ind_data = ind_series.get('atr', pd.Series(dtype=float))
    else:
        ind_label = 'Indicator'
        ind_data = pd.Series(dtype=float)

    # Header
    print(f"  {'Date':12s} {'Close':>12s} {'Signal':>8s} {ind_label:>12s} {'Regime':>12s}")
    print(f"  {'-'*12} {'-'*12} {'-'*8} {'-'*12} {'-'*12}")

    # Last N days
    last_n = close.index[-HISTORY_DAYS:]
    for dt in last_n:
        c = close.get(dt, float('nan'))

        # Signal
        entry_today = entries.get(dt, False) if dt in entries.index else False
        exit_today = exits.get(dt, False) if dt in exits.index else False
        if entry_today:
            sig_str = 'BUY'
        elif exit_today:
            sig_str = 'SELL'
        else:
            sig_str = '—'

        # Indicator value
        ind_val = ind_data.get(dt, float('nan')) if dt in ind_data.index else float('nan')
        ind_str = f"{ind_val:.4f}" if not pd.isna(ind_val) else '—'

        # Regime
        reg = regime_df['regime'].get(dt, '—') if dt in regime_df.index else '—'

        print(f"  {dt.strftime('%Y-%m-%d'):12s} {c:12.2f} {sig_str:>8s} {ind_str:>12s} {reg:>12s}")

print(f"\n{'='*70}")
print("Signal history complete.")


In [ ]:
# ============================================================
# POSITION SIZING HELPER
# ============================================================
# For each active BUY signal, compute:
#   - Entry price (today's close as estimate)
#   - ATR-based stop loss (2x ATR below entry)
#   - Position size based on risk
#   - FTMO-aware checks

ATR_STOP_MULT = 2.0     # Stop loss = 2x ATR below entry
ATR_TP_MULT = 3.0       # Take profit = 3x ATR above entry
MAX_DAILY_LOSS_PCT = 0.05  # FTMO: 5% max daily loss

print("=" * 70)
print("  POSITION SIZING — Active BUY Signals")
print("=" * 70)

has_buys = False

for ticker, d in dashboard.items():
    if d['current_signal'] != 'BUY':
        continue

    has_buys = True
    name = d['name']
    entry_price = d['price']
    close = all_data[ticker]['close']
    high = all_data[ticker]['high']
    low = all_data[ticker]['low']

    # Compute ATR (14-period) for stop loss
    if talib is not None:
        atr_val = talib.ATR(high.values, low.values, close.values, timeperiod=14)[-1]
    else:
        atr_val = _atr(high, low, close, 14).iloc[-1]

    stop_loss = entry_price - (ATR_STOP_MULT * atr_val)
    take_profit = entry_price + (ATR_TP_MULT * atr_val)
    distance_to_stop = entry_price - stop_loss

    # Position size based on risk
    risk_amount = RISK_PER_TRADE * ACCOUNT_SIZE
    if distance_to_stop > 0:
        position_size = risk_amount / distance_to_stop
    else:
        position_size = 0

    position_value = position_size * entry_price

    # FTMO daily loss check
    max_daily_loss = MAX_DAILY_LOSS_PCT * ACCOUNT_SIZE
    risk_warning = ''
    if risk_amount > max_daily_loss:
        risk_warning = ' *** EXCEEDS FTMO DAILY LOSS LIMIT ***'

    # R:R ratio
    rr_ratio = (take_profit - entry_price) / distance_to_stop if distance_to_stop > 0 else 0

    print(f"\n  {name} ({ticker})")
    print(f"    Entry:       ~${entry_price:,.2f}")
    print(f"    ATR(14):      ${atr_val:,.2f}")
    print(f"    Stop Loss:    ${stop_loss:,.2f}  ({ATR_STOP_MULT:.0f}x ATR below entry)")
    print(f"    Take Profit:  ${take_profit:,.2f}  ({ATR_TP_MULT:.0f}x ATR above entry)")
    print(f"    R:R Ratio:    1:{rr_ratio:.1f}")
    print(f"    Distance:     ${distance_to_stop:,.2f}")
    print(f"    Size:         {position_size:,.4f} units  (${position_value:,.0f} notional)")
    print(f"    Risk:         ${risk_amount:,.0f} ({RISK_PER_TRADE:.0%} of account){risk_warning}")

if not has_buys:
    print("\n  No active BUY signals today. Nothing to size.")

print("\n" + "=" * 70)
print(f"  FTMO Safety: Max daily loss = ${MAX_DAILY_LOSS_PCT * ACCOUNT_SIZE:,.0f} ({MAX_DAILY_LOSS_PCT:.0%})")
print(f"  Per-trade risk: ${RISK_PER_TRADE * ACCOUNT_SIZE:,.0f} ({RISK_PER_TRADE:.0%})")
print("=" * 70)


In [ ]:
# ============================================================
# PORTFOLIO P&L TRACKER
# ============================================================
# If you're tracking open positions, paste them here:

OPEN_POSITIONS = {
    # 'ticker': {'entry_date': '2026-03-01', 'entry_price': 18500, 'size': 1.0},
}

# For each open position, calculate unrealized P&L
if OPEN_POSITIONS:
    print("OPEN POSITION P&L:")
    total_pnl = 0
    for ticker, pos in OPEN_POSITIONS.items():
        if ticker in all_data:
            current = all_data[ticker]['close'].iloc[-1]
            entry = pos['entry_price']
            pnl_pct = (current - entry) / entry
            pnl_usd = pnl_pct * (ACCOUNT_SIZE / len(PORTFOLIO))
            total_pnl += pnl_usd
            name = PORTFOLIO.get(ticker, {}).get('name', ticker)
            emoji = '+' if pnl_pct > 0 else ''
            print(f"  {name:12s} | Entry: {entry:.2f} | Current: {current:.2f} | P&L: {emoji}{pnl_pct:.2%} (${pnl_usd:+,.0f})")
    print(f"\n  Total unrealized P&L: ${total_pnl:+,.0f} ({total_pnl/ACCOUNT_SIZE:+.2%} of account)")
else:
    print("No open positions tracked. Add entries to OPEN_POSITIONS dict above.")
    print("Example:")
    print("  OPEN_POSITIONS = {")
    print("      '^IXIC': {'entry_date': '2026-03-01', 'entry_price': 18500, 'size': 1.0},")
    print("  }")
